<a href="https://colab.research.google.com/github/skaviyashree/Sign-Language-to-Audio-Translator/blob/main/signal_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Day 1

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# --- Configuration ---
# Assuming CSV files are in the current working directory: C:\Users\Vanitha\signal_processing
TRAIN_FILE = 'sign_mnist_train.csv'
TEST_FILE = 'sign_mnist_test.csv'

# --- Data Processing ---
print(f"Starting data processing for Sign Language MNIST...")

def load_and_process_csv(filepath, is_test=False):
    """Loads CSV data, extracts labels and features, and normalizes pixels."""
    if not os.path.exists(filepath):
        print(f"Error: Required file not found at {filepath}. Please ensure '{os.path.basename(filepath)}' is in your directory.")
        return None, None

    # Read the CSV file
    df = pd.read_csv(filepath)

    # Extract Labels (The sign corresponding to the image)
    # The 'label' column is the first column in the CSV for the training data.
    if not is_test:
        labels = df['label'].values
        # Drop the label column to get only pixel features
        features = df.drop('label', axis=1).values
    else:
        # The test file may not have labels, but this dataset does.
        labels = df['label'].values
        features = df.drop('label', axis=1).values

    # Normalize Features (Pixel values are 0-255, scale them to 0-1)
    # This is critical for CNN training performance.
    features = features / 255.0

    # Reshape features to be suitable for the Convolutional Neural Network (CNN)
    # The original images are 28x28 pixels.
    # Features shape should be (N_samples, 28, 28, 1)
    N_SAMPLES = features.shape[0]
    features = features.reshape(N_SAMPLES, 28, 28, 1)

    print(f"Loaded {N_SAMPLES} samples from {os.path.basename(filepath)}. Feature shape: {features.shape}")
    return features, labels

# Load Training Data (Will be split into train/validation sets later)
X_train_full, y_train_full = load_and_process_csv(TRAIN_FILE, is_test=False)

# Load Testing Data (Will remain as a final test set)
X_test, y_test = load_and_process_csv(TEST_FILE, is_test=True)

if X_train_full is not None and X_test is not None:
    # Split the full training data into actual training and validation sets
    # We use 80% for training and 20% for validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42
    )

    print("\nSplitting training data...")
    print(f"Final Training samples (X_train): {X_train.shape[0]}")
    print(f"Validation samples (X_val): {X_val.shape[0]}")
    print(f"Final Testing samples (X_test): {X_test.shape[0]}")

    #--- Save Processed Data ---
    print("\nSaving processed data files...")

    np.save('X_train.npy', X_train)
    np.save('y_train.npy', y_train)
    np.save('X_val.npy', X_val)
    np.save('y_val.npy', y_val)
    np.save('X_test.npy', X_test)
    np.save('y_test.npy', y_test)

    print("\n==================================================")
    print("        DATA PROCESSING COMPLETE (Sign MNIST)")
    print("==================================================")
    print(f"6 Numpy files saved in: {os.getcwd()}")
    print("--- Ready for Model Training (New CNN Model Required!) ---")





Day 2 (Training the Data)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import os

# --- Configuration ---
# Data files are saved in the current working directory
X_TRAIN_FILE = 'X_train.npy'
Y_TRAIN_FILE = 'y_train.npy'
X_VAL_FILE = 'X_val.npy'
Y_VAL_FILE = 'y_val.npy'
X_TEST_FILE = 'X_test.npy'
Y_TEST_FILE = 'y_test.npy'
MODEL_FILE = 'cnn_gesture_model.h5'
NUM_CLASSES = 25  # Sign MNIST has 24 classes (A-Y, excluding J and Z)
INPUT_SHAPE = (28, 28, 1)

# --- Data Loading ---
print("Loading processed data files...")

# Check if files exist
if not all(os.path.exists(f) for f in [X_TRAIN_FILE, Y_TRAIN_FILE, X_VAL_FILE, Y_VAL_FILE, X_TEST_FILE, Y_TEST_FILE]):
    print("Error: One or more data files (.npy) not found. Please run csv_data_processor.py first.")
    # Exit gracefully if data is missing
    exit()

# Load data arrays
X_train = np.load(X_TRAIN_FILE)
y_train = np.load(Y_TRAIN_FILE)
X_val = np.load(X_VAL_FILE)
y_val = np.load(Y_VAL_FILE)
X_test = np.load(X_TEST_FILE)
y_test = np.load(Y_TEST_FILE)

# Convert labels to categorical (One-Hot Encoding)
# This is necessary for training a classification model
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)

print(f"Data loaded: Training samples={X_train.shape[0]}, Test samples={X_test.shape[0]}")
print(f"Input image shape: {INPUT_SHAPE}")
print(f"Number of classes: {NUM_CLASSES}")


# --- Model Definition (CNN) ---
def create_cnn_model():
    """Defines a Convolutional Neural Network (CNN) suitable for 28x28 images."""
    model = Sequential([
        # 1. Convolutional Layer: Learns basic features (edges, corners)
        Conv2D(32, (3, 3), activation='relu', input_shape=INPUT_SHAPE),
        # 2. Pooling Layer: Reduces dimensionality and makes model robust to shifting
        MaxPooling2D((2, 2)),

        # 3. Second Convolutional Block
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        # 4. Flatten Layer: Converts 2D feature maps into a 1D vector for the Dense layers
        Flatten(),

        # 5. Dense Layer: High-level reasoning
        Dense(128, activation='relu'),
        # 6. Dropout: Prevents overfitting by randomly setting 50% of inputs to zero
        Dropout(0.5),

        # 7. Output Layer: 24 neurons (one for each class) with softmax for probability distribution
        Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

model = create_cnn_model()
model.summary()

# --- Model Training ---
print("\n--- Starting CNN Model Training ---")

history = model.fit(
    X_train, y_train_cat,
    epochs=10,  # 10 epochs is sufficient for high accuracy on this clean dataset
    batch_size=32,
    validation_data=(X_val, y_val_cat),
    verbose=1
)

# --- Model Evaluation and Saving ---
loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

model.save(MODEL_FILE)

print("\n==================================================")
print("          CNN MODEL TRAINING COMPLETE")
print("==================================================")
print(f"Final Model saved as: {MODEL_FILE}")
print(f"Test Accuracy: {accuracy:.4f}")
print("--- Ready for Day 5: Real-Time Inference! ---")



Day 3

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import os
import winsound # <-- Built-in Windows library for audio playback
import time

# --- Configuration ---
MODEL_FILE = 'cnn_gesture_model.h5'
INPUT_SIZE = 28
AUDIO_FOLDER = "" # Looks in the current directory (where the script is run)

# Map model output index (0-24) to the actual sign (A-Y, skipping J & Z, plus UNKNOWN)
SIGN_CLASSES = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S',
    'T', 'U', 'V', 'W', 'X', 'Y', 'UNKNOWN'
]

# The audio mapping connects the class name to the file name.
# NOTE: Filenames MUST be case-sensitive (e.g., audio_A.wav)
AUDIO_MAPPING = {
    'A': 'audio_A.wav', 'B': 'audio_B.wav', 'C': 'audio_C.wav', 'D': 'audio_D.wav',
    'E': 'audio_E.wav', 'F': 'audio_F.wav', 'G': 'audio_G.wav', 'H': 'audio_H.wav',
    'I': 'audio_I.wav',
    'K': 'audio_K.wav', 'L': 'audio_L.wav', 'M': 'audio_M.wav', 'N': 'audio_N.wav',
    'O': 'audio_O.wav', 'P': 'audio_P.wav', 'Q': 'audio_Q.wav', 'R': 'audio_R.wav',
    'S': 'audio_S.wav',
    'T': 'audio_T.wav', 'U': 'audio_U.wav', 'V': 'audio_V.wav', 'W': 'audio_W.wav',
    'X': 'audio_X.wav', 'Y': 'audio_Y.wav',
    'UNKNOWN': 'audio_unknown.wav'
}

# --- Audio Playback ---
LAST_PLAYED_SIGN = ""
LAST_PLAY_TIME = 0
PLAYBACK_DELAY = 1.5  # Wait 1.5 seconds before playing the same audio again

def play_gesture_audio(gesture_name):
    """Plays the corresponding WAV file for the recognized gesture using winsound."""
    global LAST_PLAYED_SIGN, LAST_PLAY_TIME
    current_time = time.time()

    # Debouncing: Only play audio if enough time has passed AND the sign is new
    if gesture_name == LAST_PLAYED_SIGN and (current_time - LAST_PLAY_TIME < PLAYBACK_DELAY):
        return

    audio_file_name = AUDIO_MAPPING.get(gesture_name)
    if audio_file_name:
        try:
            audio_path = os.path.join(os.getcwd(), AUDIO_FOLDER, audio_file_name)
            if os.path.exists(audio_path):
                # Use winsound.PlaySound for reliable, non-blocking playback of WAV files
                # SND_FILENAME specifies the file; SND_ASYNC means the code doesn't freeze while playing.
                winsound.PlaySound(audio_path, winsound.SND_FILENAME | winsound.SND_ASYNC)
                LAST_PLAY_TIME = current_time
                LAST_PLAYED_SIGN = gesture_name
            # else: print(f"Audio file not found: {audio_path}")
        except Exception as e:
            # print(f"Error playing audio: {e}. Check WAV file format.")
            pass

# --- Model Loading ---
try:
    model = tf.keras.models.load_model('cnn_gesture_model.h5', compile=False)
    print(f"Model loaded successfully: cnn_gesture_model.h5")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure 'cnn_gesture_model.h5' is in the current directory.")
    exit()

# --- Main Recognition Loop ---
def run_real_time_recognition():
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Define ROI (Region of Interest) for the Hand
        h, w, _ = frame.shape
        roi_size = min(h, w) // 2
        top = (h - roi_size) // 2
        bottom = top + roi_size
        left = (w - roi_size) // 2
        right = left + roi_size

        # Crop the frame to the ROI
        roi = frame[top:bottom, left:right]

        # 2. Preprocessing for CNN (28x28 grayscale)
        gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        resized_roi = cv2.resize(gray_roi, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_AREA)

        # Normalize and Reshape for Model Prediction
        input_data = resized_roi / 255.0
        input_data = input_data.reshape(1, INPUT_SIZE, INPUT_SIZE, 1)

        # 3. Model Prediction
        prediction = model.predict(input_data, verbose=0)[0]
        predicted_index = np.argmax(prediction)
        confidence = prediction[predicted_index]

        # 4. Determine Output
        predicted_gesture = SIGN_CLASSES[predicted_index]
        confidence_percent = confidence * 100

        CONFIDENCE_THRESHOLD = 0.70

        if confidence_percent < CONFIDENCE_THRESHOLD * 100:
            display_text = f"Low Confidence: <{CONFIDENCE_THRESHOLD*100:.0f}%"
            recognized_sign = "WAIT"
        else:
            recognized_sign = predicted_gesture
            display_text = f"Sign: {recognized_sign} ({confidence_percent:.2f}%)"

            # 5. Audio Output
            if recognized_sign != "WAIT" and recognized_sign != "UNKNOWN":
                play_gesture_audio(recognized_sign)

        # 6. Display Output
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, display_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.imshow('Sign Language Recognizer', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_real_time_recognition()
```eof

### Your Final Step

1.  **Copy and Save:** Ensure the code above is the *only* content in your **`cnn_recognizer.py`** file.
2.  **Run the Project:** Execute the file from your terminal:

    ```bash
    python cnn_recognizer.py
    ```

This must be the final working solution that plays your WAV files directly using the Windows operating system!

Day-3(with sample checking out of 4)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import os
import winsound # Built-in Windows library for audio playback
import time
from collections import deque # Used for the prediction history buffer

# --- Configuration ---
MODEL_FILE = 'cnn_gesture_model.h5'
INPUT_SIZE = 28
AUDIO_FOLDER = "" # Looks in the current directory (where the script is run)

# Map model output index (0-24) to the actual sign (A-Y, skipping J & Z, plus UNKNOWN)
SIGN_CLASSES = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S',
    'T', 'U', 'V', 'W', 'X', 'Y', 'UNKNOWN'
]

# The audio mapping connects the class name to the file name.
AUDIO_MAPPING = {
    'A': 'audio_A.wav', 'B': 'audio_B.wav', 'C': 'audio_C.wav', 'D': 'audio_D.wav',
    'E': 'audio_E.wav', 'F': 'audio_F.wav', 'G': 'audio_G.wav', 'H': 'audio_H.wav',
    'I': 'audio_I.wav',
    'K': 'audio_K.wav', 'L': 'audio_L.wav', 'M': 'audio_M.wav', 'N': 'audio_N.wav',
    'O': 'audio_O.wav', 'P': 'audio_P.wav', 'Q': 'audio_Q.wav', 'R': 'audio_R.wav',
    'S': 'audio_S.wav',
    'T': 'audio_T.wav', 'U': 'audio_U.wav', 'V': 'audio_V.wav', 'W': 'audio_W.wav',
    'X': 'audio_X.wav', 'Y': 'audio_Y.wav',
    'UNKNOWN': 'audio_unknown.wav'
}

# --- Audio Playback ---
LAST_PLAYED_SIGN = ""
LAST_PLAY_TIME = 0
PLAYBACK_DELAY = 1.5  # Wait 1.5 seconds before playing the same audio again

def play_gesture_audio(gesture_name):
    """Plays the corresponding WAV file for the recognized gesture using winsound."""
    global LAST_PLAYED_SIGN, LAST_PLAY_TIME
    current_time = time.time()

    if gesture_name == LAST_PLAYED_SIGN and (current_time - LAST_PLAY_TIME < PLAYBACK_DELAY):
        return

    audio_file_name = AUDIO_MAPPING.get(gesture_name)
    if audio_file_name:
        try:
            audio_path = os.path.join(os.getcwd(), AUDIO_FOLDER, audio_file_name)
            if os.path.exists(audio_path):
                winsound.PlaySound(audio_path, winsound.SND_FILENAME | winsound.SND_ASYNC)
                LAST_PLAY_TIME = current_time
                LAST_PLAYED_SIGN = gesture_name
            # else: print(f"Audio file not found: {audio_path}")
        except Exception as e:
            # print(f"Error playing audio: {e}. Check WAV file format.")
            pass

# --- Model Loading ---
try:
    model = tf.keras.models.load_model('cnn_gesture_model.h5', compile=False)
    print(f"Model loaded successfully: cnn_gesture_model.h5")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure 'cnn_gesture_model.h5' is in the current directory.")
    exit()

# --- Main Recognition Loop ---
def run_real_time_recognition():
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    # --- STABILIZATION SETUP ---
    PREDICTION_HISTORY_LENGTH = 5 # Store last 5 predictions
    MIN_VOTES_REQUIRED = 4        # Need 4 out of 5 for a stable classification
    history = deque(maxlen=PREDICTION_HISTORY_LENGTH)
    # --- END STABILIZATION SETUP ---

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Define ROI (Region of Interest) and Preprocessing (28x28 grayscale)
        h, w, _ = frame.shape
        roi_size = min(h, w) // 2
        top = (h - roi_size) // 2
        bottom = top + roi_size
        left = (w - roi_size) // 2
        right = left + roi_size

        roi = frame[top:bottom, left:right]
        gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        resized_roi = cv2.resize(gray_roi, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_AREA)

        # Normalize and Reshape for Model Prediction
        input_data = resized_roi / 255.0
        input_data = input_data.reshape(1, INPUT_SIZE, INPUT_SIZE, 1)

        # 2. Model Prediction
        prediction = model.predict(input_data, verbose=0)[0]
        predicted_index = np.argmax(prediction)
        confidence = prediction[predicted_index]

        predicted_gesture_raw = SIGN_CLASSES[predicted_index]
        confidence_percent = confidence * 100

        CONFIDENCE_THRESHOLD = 0.70

        if confidence_percent < CONFIDENCE_THRESHOLD * 100:
            # If low confidence, treat as "WAIT" (no prediction)
            current_prediction_stable = "WAIT"
        else:
            current_prediction_stable = predicted_gesture_raw

        # 3. Stabilization Logic
        history.append(current_prediction_stable)

        # Check for stable prediction (majority vote)
        if len(history) == PREDICTION_HISTORY_LENGTH:
            from collections import Counter
            counts = Counter(history)
            most_common, count = counts.most_common(1)[0]

            if count >= MIN_VOTES_REQUIRED and most_common != "WAIT":
                recognized_sign = most_common
                display_text = f"Sign: {recognized_sign} (STABLE)"

                # 4. Audio Output (Plays only when stable)
                play_gesture_audio(recognized_sign)
            else:
                recognized_sign = "..."
                display_text = f"Waiting for stability ({count}/{MIN_VOTES_REQUIRED})"
        else:
            recognized_sign = "..."
            display_text = "Initializing..."


        # 5. Display Output
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, display_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.imshow('Sign Language Recognizer', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_real_time_recognition()


### Your Final Step

1.  **Copy and Save:** Ensure the code above is the *only* content in your **`cnn_recognizer.py`** file.
2.  **Run the Project:** Execute the file: `python cnn_recognizer.py`

This must be the final working solution! The history buffer should force the model to be stable before playing any sound or changing the display, fixing your flickering issue.

Day -3 (checking samples out of 8 or 10)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import os
import winsound
import time
from collections import deque
from collections import Counter

# --- Configuration ---
MODEL_FILE = 'cnn_gesture_model.h5'
INPUT_SIZE = 28
AUDIO_FOLDER = ""

# Map model output index (0-24) to the actual sign (A-Y, skipping J & Z, plus UNKNOWN)
SIGN_CLASSES = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I',
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S',
    'T', 'U', 'V', 'W', 'X', 'Y', 'UNKNOWN'
]

# The audio mapping connects the class name to the file name.
AUDIO_MAPPING = {
    'A': 'audio_A.wav', 'B': 'audio_B.wav', 'C': 'audio_C.wav', 'D': 'audio_D.wav',
    'E': 'audio_E.wav', 'F': 'audio_F.wav', 'G': 'audio_G.wav', 'H': 'audio_H.wav',
    'I': 'audio_I.wav',
    'K': 'audio_K.wav', 'L': 'audio_L.wav', 'M': 'audio_M.wav', 'N': 'audio_N.wav',
    'O': 'audio_O.wav', 'P': 'audio_P.wav', 'Q': 'audio_Q.wav', 'R': 'audio_R.wav',
    'S': 'audio_S.wav',
    'T': 'audio_T.wav', 'U': 'audio_U.wav', 'V': 'audio_V.wav', 'W': 'audio_W.wav',
    'X': 'audio_X.wav', 'Y': 'audio_Y.wav',
    'UNKNOWN': 'audio_unknown.wav'
}

# --- Audio Playback ---
LAST_PLAYED_SIGN = ""
LAST_PLAY_TIME = 0
PLAYBACK_DELAY = 1.5

def play_gesture_audio(gesture_name):
    """Plays the corresponding WAV file for the recognized gesture using winsound."""
    global LAST_PLAYED_SIGN, LAST_PLAY_TIME
    current_time = time.time()

    if gesture_name == LAST_PLAYED_SIGN and (current_time - LAST_PLAY_TIME < PLAYBACK_DELAY):
        return

    audio_file_name = AUDIO_MAPPING.get(gesture_name)
    if audio_file_name:
        try:
            audio_path = os.path.join(os.getcwd(), audio_file_name) # Removed AUDIO_FOLDER
            if os.path.exists(audio_path):
                winsound.PlaySound(audio_path, winsound.SND_FILENAME | winsound.SND_ASYNC)
                LAST_PLAY_TIME = current_time
                LAST_PLAYED_SIGN = gesture_name
        except Exception as e:
            pass

# --- Model Loading ---
try:
    model = tf.keras.models.load_model('cnn_gesture_model.h5', compile=False)
    print(f"Model loaded successfully: cnn_gesture_model.h5")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure 'cnn_gesture_model.h5' is in the current directory.")
    exit()

# --- Main Recognition Loop ---
def run_real_time_recognition():
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    # --- STABILIZATION SETUP ---
    PREDICTION_HISTORY_LENGTH = 10 # Increased history
    MIN_VOTES_REQUIRED = 8        # Increased required votes (80% majority)
    history = deque(maxlen=PREDICTION_HISTORY_LENGTH)
    # --- END STABILIZATION SETUP ---

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Define ROI (Region of Interest) - ADJUSTED FOR BETTER CENTERING
        h, w, _ = frame.shape

        # New: Reduced ROI size to focus closer on the hand (e.g., 40% of the screen)
        roi_size = min(h, w) // 2

        # Calculate margins to center the ROI
        margin_x = (w - roi_size) // 2
        margin_y = (h - roi_size) // 2

        # Define coordinates for the ROI (slightly tighter area)
        top = margin_y
        bottom = margin_y + roi_size
        left = margin_x
        right = margin_x + roi_size

        # Crop the frame
        roi = frame[top:bottom, left:right]

        # 2. Preprocessing for CNN (28x28 grayscale)
        gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        resized_roi = cv2.resize(gray_roi, (INPUT_SIZE, INPUT_SIZE), interpolation=cv2.INTER_AREA)

        # Normalize and Reshape for Model Prediction
        input_data = resized_roi / 255.0
        input_data = input_data.reshape(1, INPUT_SIZE, INPUT_SIZE, 1)

        # 3. Model Prediction
        prediction = model.predict(input_data, verbose=0)[0]
        predicted_index = np.argmax(prediction)
        confidence = prediction[predicted_index]

        predicted_gesture_raw = SIGN_CLASSES[predicted_index]
        confidence_percent = confidence * 100

        CONFIDENCE_THRESHOLD = 0.70

        if confidence_percent < CONFIDENCE_THRESHOLD * 100:
            # If low confidence, treat as "WAIT" (no prediction)
            current_prediction_stable = "WAIT"
        else:
            current_prediction_stable = predicted_gesture_raw

        # 4. Stabilization Logic
        history.append(current_prediction_stable)

        recognized_sign = "..."
        display_text = "Initializing..."

        # Check for stable prediction (majority vote)
        if len(history) == PREDICTION_HISTORY_LENGTH:
            counts = Counter(history)
            most_common, count = counts.most_common(1)[0]

            if count >= MIN_VOTES_REQUIRED and most_common != "WAIT":
                # Stable and high-confidence prediction found
                recognized_sign = most_common
                display_text = f"Sign: {recognized_sign} (STABLE {count}/{PREDICTION_HISTORY_LENGTH})"

                # 5. Audio Output (Plays only when stable)
                play_gesture_audio(recognized_sign)
            else:
                # Prediction is unstable or below confidence, or waiting for history buffer
                display_text = f"Waiting for stability ({count}/{MIN_VOTES_REQUIRED})"


        # 6. Display Output
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, display_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.imshow('Sign Language Recognizer', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    run_real_time_recognition()


### Your Final Step

1.  **Copy and Save:** Ensure the code above is the *only* content in your **`cnn_recognizer.py`** file.
2.  **Run the Project:** Execute the file: `python cnn_recognizer.py`

This enhanced stabilization logic should give you solid, reliable predictions. Try making the signs so they **fill the green box** completely!